# dazpy — Interactive Workspace

Connect Python to a live DAZ Studio session and explore the scene, control figures, and trigger renders.

## Prerequisites

- DAZ Studio is running with the **DazScriptServer** plugin active
- Default address: `http://127.0.0.1:18811`
- Verify with: `curl http://127.0.0.1:18811/health`

## API Documentation

| Section | Link |
|---|---|
| Overview | [bluemoonfoundry.github.io/daz-script-server/](https://bluemoonfoundry.github.io/daz-script-server/) |
| Quick Start | [quickstart](https://bluemoonfoundry.github.io/daz-script-server/quickstart.html) |
| DazClient | [api/client](https://bluemoonfoundry.github.io/daz-script-server/api/client.html) |
| DazScene | [api/scene](https://bluemoonfoundry.github.io/daz-script-server/api/scene.html) |
| DazSkeleton / DazBone | [api/skeleton](https://bluemoonfoundry.github.io/daz-script-server/api/skeleton.html) |
| DazNode | [api/nodes](https://bluemoonfoundry.github.io/daz-script-server/api/nodes.html) |
| DazCamera | [api/camera](https://bluemoonfoundry.github.io/daz-script-server/api/camera.html) |
| DazLight | [api/light](https://bluemoonfoundry.github.io/daz-script-server/api/light.html) |
| DazMaterial | [api/materials](https://bluemoonfoundry.github.io/daz-script-server/api/materials.html) |
| DazRenderSettings | [api/render](https://bluemoonfoundry.github.io/daz-script-server/api/render.html) |
| DazTimeline / DazAnimation | [api/timeline](https://bluemoonfoundry.github.io/daz-script-server/api/timeline.html) |
| DazPose | [api/pose](https://bluemoonfoundry.github.io/daz-script-server/api/pose.html) |
| DazGeometry | [api/geometry](https://bluemoonfoundry.github.io/daz-script-server/api/geometry.html) |
| DazProperty | [api/properties](https://bluemoonfoundry.github.io/daz-script-server/api/properties.html) |
| ExecutionResult | [api/result](https://bluemoonfoundry.github.io/daz-script-server/api/result.html) |
| Exceptions | [api/exceptions](https://bluemoonfoundry.github.io/daz-script-server/api/exceptions.html) |
| Vec3 / Quat / BoundingBox | [api/math3](https://bluemoonfoundry.github.io/daz-script-server/api/math3.html) |
| Batch | [api/batch](https://bluemoonfoundry.github.io/daz-script-server/api/batch.html) |

## Quick Reference

### Key classes

| Class | Import | Purpose |
|---|---|---|
| `DazClient` | `dazpy` | Low-level HTTP client — `execute()`, `health()`, `metrics()` |
| `DazScene` | `dazpy` | Scene-level queries — nodes, skeletons, cameras, frame, undo |
| `DazSkeleton` | `dazpy` | Figure with bones — `find_bone()`, `bones()`, `apply_pose()` |
| `DazBone` | `dazpy` | Individual bone — `set_local_rotation()`, `local_euler` |
| `DazNode` | `dazpy` | Generic scene node — `label`, `position`, `visible` |
| `DazCamera` | `dazpy` | Camera node — `focal_length`, `set_active()` |
| `DazLight` | `dazpy` | Light node — `intensity`, `color` |
| `DazMaterial` | `dazpy` | Material on a node — `diffuse_color`, `set_property()` |
| `DazRenderSettings` | `dazpy` | Active render settings — `width`, `height`, `engine` |
| `DazTimeline` | `dazpy` | Playback range, FPS, current frame |
| `DazPose` | `dazpy` | Captured pose snapshot — save / restore bone rotations |
| `DazGeometry` | `dazpy` | Mesh data — vertex count, face count, UVs |
| `Batch` | `dazpy` | Fan-out multiple operations in fewer HTTP round-trips |

### DazScript rules (inline scripts via `client.execute()`)

- **No top-level `return`** — the result is the value of the last expression: `App.version;`
- **Multi-statement scripts need an IIFE**: wrap in `(function() { ...; return value; })()`  
  `ScriptBuilder.iife(code)` does this for you when working with the SDK
- **Result lives in `.value`**, not `.result`: `r = client.execute(script); r.value`
- **Print via `print()`** in DazScript; output appears in `result.output` (list of strings)

### Example scripts

| Topic | Path |
|---|---|
| Raw script / primary selection | `docs/examples/fundamentals/raw_script.py` |
| Full scene inventory (JSON) | `docs/examples/fundamentals/scene_introspection.py` |
| Scene node inventory | `docs/examples/fundamentals/scene_inventory.py` |
| Save scene copy | `docs/examples/fundamentals/scene_save_copy.py` |
| Pose transfer between figures | `docs/examples/character/pose_transfer.py` |
| Character state dump | `docs/examples/character/character_state.py` |
| Batch render morph variations | `docs/examples/rendering/batch_render_morph_variations.py` |
| Turntable render | `docs/examples/rendering/turntable.py` |
| Multi-camera render | `docs/examples/rendering/multi_camera_render.py` |
| Body measurements | `docs/examples/geometry/body_measurements.py` |
| BVH motion import | `docs/examples/bvh/bvh_import.py` |
| USD export | `docs/examples/export/scene_to_usd.py` |

In [8]:
import dazpy
from dazpy import DazClient, DazScene

client = DazClient()  # default: 127.0.0.1:18811
print(f"dazpy {dazpy.__version__} — connected to {client._base}")

dazpy 2.6.0 — connected to http://127.0.0.1:18811


---
## 1. Server health

In [9]:
import pprint
pprint.pprint(client.health())

{'active_requests': 0,
 'auth_enabled': False,
 'running': True,
 'status': 'ok',
 'uptime_seconds': 14232,
 'version': '2.6.0'}


---
## 2. Scene overview

In [10]:
scene = DazScene(client)
print(f"{scene.num_nodes()} nodes  |  {scene.num_skeletons()} skeletons  |  frame {scene.frame()}")
print()
for n in scene.nodes():
    print(f"  {type(n).__name__:<14}  {n.name}")

502 nodes  |  11 skeletons  |  frame 0

  DazNode         Tonemapper Options
  DazNode         Environment Options
  DazSkeleton     Genesis9
  DazNode         hip
  DazNode         pelvis
  DazNode         l_thigh
  DazNode         l_shin
  DazNode         l_foot
  DazNode         l_toes
  DazNode         l_bigtoe1
  DazNode         l_bigtoe2
  DazNode         l_indextoe1
  DazNode         l_indextoe2
  DazNode         l_midtoe1
  DazNode         l_midtoe2
  DazNode         l_ringtoe1
  DazNode         l_ringtoe2
  DazNode         l_pinkytoe1
  DazNode         l_pinkytoe2
  DazNode         l_metatarsal
  DazNode         l_thightwist1
  DazNode         l_thightwist2
  DazNode         r_thigh
  DazNode         r_shin
  DazNode         r_foot
  DazNode         r_toes
  DazNode         r_bigtoe1
  DazNode         r_bigtoe2
  DazNode         r_indextoe1
  DazNode         r_indextoe2
  DazNode         r_midtoe1
  DazNode         r_midtoe2
  DazNode         r_ringtoe1
  DazNode         r_rin

---
## 3. Find a figure and read a bone

`scene.find_skeleton_by_label()` searches by the name shown in DAZ Studio's Scene panel.  
Adjust the label to match a figure in your scene.

In [12]:
figure = scene.find_skeleton_by_label("Agnes Montenegro")  # change to match your scene
bone = figure.find_bone("head")
print("head local euler (xyz°):", bone.local_euler)

head local euler (xyz°): (-3.49952101707458, -4.64459180831909, -0.12897689640522)


---
## 4. Raw DazScript

Use `client.execute()` to run arbitrary DazScript when the SDK doesn't cover what you need.

**Rules:**
- Single expression → no wrapper needed, just end with `;`
- Multiple statements → wrap in an IIFE: `(function() { ...; return value; })()`
- Result is in `result.value`; console output (`print()` in DazScript) is in `result.output`

In [ ]:
# Single expression — no IIFE needed
r = client.execute("App.version;")
print("DAZ Studio version (packed int):", r.value)

In [ ]:
# Multiple statements — IIFE required
r = client.execute("""
(function() {
    var node = Scene.getPrimarySelection();
    if (!node) return null;
    return {
        name:  node.getName(),
        label: node.getLabel(),
        type:  node.className()
    };
})()
""")
pprint.pprint(r.value)